In [2]:
"""
Federated Learning — FedAvg with Logistic Regression
Dataset: ULB Credit Card Fraud (creditcard.csv from Kaggle)
Simulates 5 banks as FL clients with non-IID data splits
"""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, roc_auc_score,
                             precision_score, recall_score, f1_score)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import copy, warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_BANKS        = 5      # number of federated clients
FL_ROUNDS      = 20     # communication rounds
LOCAL_EPOCHS   = 3      # local training iterations per round
RANDOM_STATE   = 42

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
print("Loading dataset...")
df = pd.read_csv("/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv")
print(f"  Total records : {len(df):,}")
print(f"  Fraud rate    : {df['Class'].mean()*100:.2f}%")

X = df.drop("Class", axis=1).values
y = df["Class"].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# ── NON-IID SPLIT ACROSS BANKS ────────────────────────────────────────────────
# Each bank gets a different fraud rate (simulates real-world non-IID)
def non_iid_split(X, y, n_banks, seed=42):
    rng = np.random.default_rng(seed)
    fraud_idx   = np.where(y == 1)[0]
    legit_idx   = np.where(y == 0)[0]

    # Dirichlet split — uneven fraud distribution per bank
    fraud_splits = rng.dirichlet(np.ones(n_banks) * 0.5) * len(fraud_idx)
    fraud_splits = np.round(fraud_splits).astype(int)
    fraud_splits[-1] = len(fraud_idx) - fraud_splits[:-1].sum()

    # Equal legit split
    legit_per_bank = len(legit_idx) // n_banks

    banks = []
    fi, li = 0, 0
    for i in range(n_banks):
        f_end = fi + fraud_splits[i]
        l_end = li + legit_per_bank
        idx = np.concatenate([fraud_idx[fi:f_end], legit_idx[li:l_end]])
        rng.shuffle(idx)
        banks.append((X[idx], y[idx]))
        print(f"  Bank {i+1}: {len(idx):>6,} records | fraud rate {y[idx].mean()*100:.2f}%")
        fi = f_end
        li = l_end
    return banks

print(f"\nSplitting data across {N_BANKS} banks (non-IID)...")
bank_data = non_iid_split(X_train, y_train, N_BANKS)

# ── FL HELPERS ────────────────────────────────────────────────────────────────
def get_model_params(model):
    return {"coef": model.coef_.copy(), "intercept": model.intercept_.copy()}

def set_model_params(model, params):
    model.coef_      = params["coef"].copy()
    model.intercept_ = params["intercept"].copy()
    return model

def fedavg(param_list, weights):
    avg = copy.deepcopy(param_list[0])
    total = sum(weights)
    for key in avg:
        avg[key] = sum(p[key] * (w / total) for p, w in zip(param_list, weights))
    return avg

def init_global_model():
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                               class_weight="balanced", solver="lbfgs")
    # Warm start requires at least one fit — use a tiny dummy fit
    X_dummy = np.random.randn(10, X_train.shape[1])
    y_dummy = np.array([0]*9 + [1])
    model.fit(X_dummy, y_dummy)
    return model

# ── FEDERATED TRAINING ────────────────────────────────────────────────────────
print(f"\nStarting FL training: {FL_ROUNDS} rounds × {N_BANKS} banks\n")
global_model = init_global_model()

for rnd in range(1, FL_ROUNDS + 1):
    global_params  = get_model_params(global_model)
    local_params   = []
    local_sizes    = []

    for bank_id, (X_b, y_b) in enumerate(bank_data):
        # SMOTE per bank to handle imbalance locally
        try:
            sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(3, y_b.sum()-1))
            X_res, y_res = sm.fit_resample(X_b, y_b)
        except Exception:
            X_res, y_res = X_b, y_b

        local_model = init_global_model()
        set_model_params(local_model, global_params)

        for _ in range(LOCAL_EPOCHS):
            local_model.fit(X_res, y_res)

        local_params.append(get_model_params(local_model))
        local_sizes.append(len(X_b))

    # Aggregate
    global_params = fedavg(local_params, local_sizes)
    set_model_params(global_model, global_params)

    # Evaluate every 5 rounds
    if rnd % 5 == 0 or rnd == 1:
        y_pred  = global_model.predict(X_test)
        y_prob  = global_model.predict_proba(X_test)[:, 1]
        auc     = roc_auc_score(y_test, y_prob)
        prec    = precision_score(y_test, y_pred, zero_division=0)
        rec     = recall_score(y_test, y_pred, zero_division=0)
        f1      = f1_score(y_test, y_pred, zero_division=0)
        print(f"  Round {rnd:>3} | AUC: {auc:.4f} | Precision: {prec:.4f} "
              f"| Recall: {rec:.4f} | F1: {f1:.4f}")

# ── FINAL EVALUATION ──────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL GLOBAL MODEL EVALUATION")
print("="*60)
y_pred = global_model.predict(X_test)
y_prob = global_model.predict_proba(X_test)[:, 1]
print(f"AUC-ROC  : {roc_auc_score(y_test, y_prob):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))

# ── PER-BANK FAIRNESS ─────────────────────────────────────────────────────────
print("Per-Bank AUC (Fairness Check):")
bank_aucs = []
for i, (X_b, y_b) in enumerate(bank_data):
    if len(np.unique(y_b)) < 2:
        continue
    prob = global_model.predict_proba(X_b)[:, 1]
    auc  = roc_auc_score(y_b, prob)
    bank_aucs.append(auc)
    print(f"  Bank {i+1}: AUC = {auc:.4f}")
print(f"  σ_AUC (fairness) = {np.std(bank_aucs):.4f}  "
      f"(lower is fairer across banks)")

Loading dataset...
  Total records : 284,807
  Fraud rate    : 0.17%

Splitting data across 5 banks (non-IID)...
  Bank 1: 45,601 records | fraud rate 0.24%
  Bank 2: 45,666 records | fraud rate 0.39%
  Bank 3: 45,491 records | fraud rate 0.00%
  Bank 4: 45,594 records | fraud rate 0.23%
  Bank 5: 45,492 records | fraud rate 0.00%

Starting FL training: 20 rounds × 5 banks

  Round   1 | AUC: 0.9557 | Precision: 0.6860 | Recall: 0.8469 | F1: 0.7580
  Round   5 | AUC: 0.9557 | Precision: 0.6860 | Recall: 0.8469 | F1: 0.7580
  Round  10 | AUC: 0.9557 | Precision: 0.6860 | Recall: 0.8469 | F1: 0.7580
  Round  15 | AUC: 0.9557 | Precision: 0.6860 | Recall: 0.8469 | F1: 0.7580
  Round  20 | AUC: 0.9557 | Precision: 0.6860 | Recall: 0.8469 | F1: 0.7580

FINAL GLOBAL MODEL EVALUATION
AUC-ROC  : 0.9557
F1 Score : 0.7580

Classification Report:
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.69      0.85      0.76